# Week 8 Lab — Figures worth showing people

**HWRS 564a · Fall 2026**

Up to now you have made plots to find out what the data says. This week you make
plots to *tell someone else* — which is a different job with different rules.

The difference is mostly not aesthetic. A figure that encodes its meaning in
colour alone is unreadable to about one person in twelve; a figure with an
unlabelled axis is unreadable to everyone. Those are correctness problems, and
they are the ones we spend the session on.

## How to use this notebook

Run each cell with **Shift+Enter**. Cells marked  **`# YOUR TURN`**  have
something for you to write. Cells marked **`# CHECK`** verify your answer — if
they run without complaint, you're right.

> **Before you submit anything all semester:** *Kernel → Restart Kernel and Run
> All Cells*. A notebook that only works when run out of order is not finished.


## Learning objectives

By the end of this notebook you can:

1. Build multi-panel figures with `subplots`, and share axes deliberately
2. Choose a colormap that suits the data, and say why perceptual uniformity matters
3. Encode a distinction redundantly, so colour is never load-bearing on its own
4. Annotate a figure so the reader's eye lands where you want it
5. Draw a defensible map from latitude and longitude
6. Save at a resolution and size that survives a journal or a projector

---

## Part 1 — Figure and axes

Everything starts the same way. `fig` is the page; `ax` is one set of axes on it.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = next(p for p in Path.cwd().parents if (p / "pyproject.toml").exists())
DATA = ROOT / "data"

wells = pd.read_csv(DATA / "tucson_basin_wells.csv", dtype={"site_no": str})
levels = pd.read_csv(DATA / "tucson_water_levels.csv",
                     dtype={"site_no": str}, parse_dates=["date"])
flow = pd.read_csv(DATA / "cache" / "nwis_09484000_dv.csv",
                   parse_dates=["datetime"]).set_index("datetime")

print(f"{len(wells):,} wells, {len(levels):,} levels, {len(flow):,} days of flow")

**Always use the object-oriented interface.** `plt.plot()` draws on whatever
axes matplotlib last touched, which is fine for one plot and a bug generator for
anything else. `ax.plot()` says exactly where the line goes.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 6))

print(type(axes), axes.shape)
axes[0, 0].set_title("axes[0, 0]")
axes[0, 1].set_title("axes[0, 1]")
axes[1, 0].set_title("axes[1, 0]")
axes[1, 1].set_title("axes[1, 1]")

fig.suptitle("subplots(2, 2) gives you a 2x2 array of axes")
plt.tight_layout()
plt.show()

### Sharing axes is a claim about comparability

`sharey=True` forces every panel onto the same scale. Do that when the panels
should be compared, and *don't* when they shouldn't — a shared axis on
incomparable quantities makes one panel look flat for no reason.

In [ ]:
sites = ["320906110592701", "320824110593001", "321208110525001"]

fig, axes = plt.subplots(1, 3, figsize=(11, 3.2), sharey=True)

for ax, site in zip(axes, sites):
    one = levels[levels["site_no"] == site].sort_values("date")
    ax.plot(one["date"], one["depth_to_water_ft"], color="#AB0520", lw=1.2)
    ax.set_title(site[-6:], fontsize=10)
    ax.grid(alpha=0.3)

axes[0].set_ylabel("depth to water (ft)")
axes[0].invert_yaxis()          # shared, so inverting one inverts all three
fig.suptitle("Three wells on a shared scale — the decline is comparable")
plt.tight_layout()
plt.show()

Because the y axis is shared, inverting `axes[0]` inverts all of them, and the
middle well's 110 ft of decline is visibly larger than the left well's 6 ft. Set
`sharey=False` and all three would look identical, which would be a lie told
entirely with axis limits.

### YOUR TURN 1

Build a two-panel figure, stacked vertically, sharing the **x** axis:

- top: daily discharge at Sabino Creek for 2006, as a line
- bottom: the same year's data as a 7-day rolling mean

Share x so the two line up in time. Label both y axes.

In [ ]:
q2006 = flow.loc["2006", "discharge_cfs"]

# YOUR TURN
fig, axes = ...

In [ ]:
# CHECK
assert len(axes) == 2, f"expected 2 panels, got {len(axes)}"
assert axes[0].get_shared_x_axes().joined(axes[0], axes[1]), "the x axes aren't shared"
assert axes[0].get_ylabel() and axes[1].get_ylabel(), "label both y axes"
assert len(axes[0].lines) >= 1 and len(axes[1].lines) >= 1, "both panels need a line"
print("Two panels, shared x, both labelled. Correct.")
plt.show()

---

## Part 2 — Colour

Three kinds of colormap, three kinds of data. Using the wrong one misleads.

| Data | Colormap | Example |
|---|---|---|
| goes one way from low to high | **sequential** | `viridis`, `Blues` |
| has a meaningful middle | **diverging** | `RdBu`, `coolwarm` |
| unordered categories | **qualitative** | `tab10` |

In [ ]:
gradient = np.linspace(0, 1, 256).reshape(1, -1)

fig, axes = plt.subplots(4, 1, figsize=(8, 2.6))
for ax, name in zip(axes, ["viridis", "RdBu", "tab10", "jet"]):
    ax.imshow(gradient, aspect="auto", cmap=name)
    ax.set_yticks([])
    ax.set_xticks([])
    ax.set_ylabel(name, rotation=0, ha="right", va="center", fontsize=10)
plt.tight_layout()
plt.show()

Look at `jet`, the bottom one. It has bright bands at cyan and yellow and dark
regions at both ends, so **equal steps in the data are not equal steps in
apparent brightness**. It invents boundaries that aren't in the data and hides
ones that are. It was the matplotlib default until 2015 and you will still see
it everywhere; don't add to the pile.

`viridis` is perceptually uniform and stays monotonic when printed in greyscale.
That is the default for a reason.

### Colour cannot be the only signal

Roughly one student per cohort — and one reader in twelve — has a colour vision
deficiency. Deuteranopia makes red and green nearly identical.

In [ ]:
years = np.arange(2005, 2025)
rng = np.random.default_rng(564)
series_a = 40 + np.cumsum(rng.normal(1.2, 2.0, len(years)))
series_b = 40 + np.cumsum(rng.normal(0.3, 2.0, len(years)))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4), sharey=True)

# colour only
axes[0].plot(years, series_a, color="#D62728", lw=2, label="heavily pumped")
axes[0].plot(years, series_b, color="#2CA02C", lw=2, label="lightly pumped")
axes[0].set_title("colour only — indistinguishable to some readers")

# colour AND dash pattern AND marker
axes[1].plot(years, series_a, color="#AB0520", lw=2, ls="-", marker="o", ms=4,
             label="heavily pumped")
axes[1].plot(years, series_b, color="#0C234B", lw=2, ls="--", marker="s", ms=4,
             label="lightly pumped")
axes[1].set_title("colour + dash + marker — readable either way")

for ax in axes:
    ax.legend(frameon=False, fontsize=9)
    ax.set_xlabel("year")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("depth to water (ft)")
plt.tight_layout()
plt.show()

The right panel costs two extra keyword arguments. **Encode every distinction at
least twice** — colour plus dash, or colour plus marker — and the figure works in
greyscale, on a bad projector, and for everyone in the room.

### YOUR TURN 2

Make a scatter of well depth against land surface elevation, coloured by
`well_depth_ft` with a **sequential, perceptually uniform** colormap, and add a
colorbar with a label.

Store the scatter in `sc` so the check can inspect it.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))

# YOUR TURN
sc = ...

fig.colorbar(sc, ax=ax, label="well depth (ft)")
ax.set_xlabel("land surface elevation (ft)")
ax.set_ylabel("well depth (ft)")
plt.tight_layout()
plt.show()

In [ ]:
# CHECK
assert sc is not None, "assign ax.scatter(...) to sc"
assert sc.get_array() is not None, "pass c=... so colour carries the depth"
cmap_name = sc.get_cmap().name
assert cmap_name in {"viridis", "plasma", "inferno", "magma", "cividis"}, (
    f"{cmap_name!r} is not perceptually uniform — use viridis or a sibling"
)
print(f"{sc.get_offsets().shape[0]:,} wells, coloured with {cmap_name}. Correct.")

---

## Part 3 — Annotation

A reader looks at a figure for a few seconds. Annotation is how you spend those
seconds on the thing you care about, instead of leaving them to search.

In [ ]:
one = levels[levels["site_no"] == "320824110593001"].sort_values("date")

fig, ax = plt.subplots(figsize=(9.5, 4))
ax.plot(one["date"], one["depth_to_water_ft"], color="#AB0520", lw=1.5)
ax.invert_yaxis()

# a horizontal reference line, labelled
ax.axhline(one["depth_to_water_ft"].iloc[0], color="grey", ls=":", lw=1)
ax.text(one["date"].iloc[2], one["depth_to_water_ft"].iloc[0] - 4,
        "level when records began", fontsize=9, color="grey")

# a shaded era — the mid-century agricultural pumping peak
ax.axvspan(pd.Timestamp("1950-01-01"), pd.Timestamp("1980-01-01"),
           color="#81D3EB", alpha=0.18)
ax.text(pd.Timestamp("1952-01-01"), 30, "peak agricultural pumping", fontsize=9,
        color="#1E5288")

# an arrow to the deepest point
deepest = one.loc[one["depth_to_water_ft"].idxmax()]
ax.annotate(f"{deepest['depth_to_water_ft']:.0f} ft",
            xy=(deepest["date"], deepest["depth_to_water_ft"]),
            xytext=(-70, 26), textcoords="offset points", fontsize=9,
            arrowprops=dict(arrowstyle="->", color="#0C234B", lw=1.2))

ax.set_ylabel("depth to water (ft below land surface)")
ax.set_title("Well D-15-13 11CBA: 110 ft of decline over 63 years of record")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Four annotations, four jobs: a baseline to measure against, a shaded period to
give context, an arrow to the extreme, and a title that states the finding
rather than naming the variables.

**Titles are the most wasted line in scientific figures.** "Depth to water vs.
time" tells the reader what they can already see. Say what it shows.

### YOUR TURN 3

Take the monthly climatology from last week and mark the monsoon.

- Shade June through September with `axvspan`
- Add a text label saying `"monsoon"` inside the shading
- Give the figure a title that states a *finding*, not a variable list

In [ ]:
q = flow["discharge_cfs"]
climatology = q.groupby(q.index.month).mean()

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.bar(climatology.index, climatology.values, color="#1E5288", alpha=0.85)
ax.set_xticks(range(1, 13))
ax.set_xlabel("month")
ax.set_ylabel("mean daily discharge (cfs)")

# YOUR TURN
...

In [ ]:
# CHECK
assert len(ax.patches) > 12, "axvspan adds a patch — did you call it?"
assert any("monsoon" in t.get_text().lower() for t in ax.texts), \
    "add a text label saying 'monsoon'"
title = ax.get_title()
assert title, "give the figure a title"
assert "vs" not in title.lower() and len(title.split()) >= 4, \
    "say what the figure shows, not which variables are on which axis"
print(f"title: {title!r}")
print("Correct.")
plt.show()

---

## Part 4 — Maps

A map is a scatter plot with two extra obligations: the aspect ratio has to be
right, and the reader has to be able to tell where they are.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))

# wrong: default aspect stretches the basin
axes[0].scatter(wells["longitude"], wells["latitude"], s=4, c="#AB0520", alpha=0.5)
axes[0].set_title("default aspect — the basin is distorted")

# right: one degree of latitude is not one degree of longitude
axes[1].scatter(wells["longitude"], wells["latitude"], s=4, c="#AB0520", alpha=0.5)
axes[1].set_aspect(1 / np.cos(np.deg2rad(wells["latitude"].mean())))
axes[1].set_title("corrected for latitude")

for ax in axes:
    ax.set_xlabel("longitude")
    ax.set_ylabel("latitude")
plt.tight_layout()
plt.show()

At 32°N a degree of longitude is only `cos(32°) ≈ 0.85` as long as a degree of
latitude. Plotting them 1:1 stretches everything east-west by 18%, which is
enough to change what a reader thinks the basin's shape is.

For a small area, that aspect correction is honest and sufficient. For anything
larger — or when you need a real projection, a scale bar, or a basemap —
`cartopy` does it properly:

In [ ]:
import cartopy.crs as ccrs

fig = plt.figure(figsize=(6.5, 6.5))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())

sc = ax.scatter(wells["longitude"], wells["latitude"],
                c=wells["well_depth_ft"], s=10, cmap="viridis",
                transform=ccrs.PlateCarree())

gl = ax.gridlines(draw_labels=True, alpha=0.3, linestyle=":")
gl.top_labels = gl.right_labels = False
fig.colorbar(sc, ax=ax, label="well depth (ft)", shrink=0.75)
ax.set_title("Tucson basin monitoring wells")
# NOTE: no plt.tight_layout() here — see below.
plt.show()

> **Two things about cartopy that will cost you an afternoon.**
>
> **`plt.tight_layout()` breaks it.** On a cartopy `GeoAxes` it raises
> `GEOSException: Points of LinearRing do not form a closed linestring` from
> deep inside shapely, and nothing in that message mentions the layout call that
> caused it. Use `fig.subplots_adjust()` or `constrained_layout=True` instead.
>
> **Coastlines and basemaps need the internet.** `ax.coastlines()`,
> `ax.add_feature(...)`, and any basemap download shapefiles from Natural Earth
> on first use. That works on your laptop and fails in a fresh codespace with no
> network. The projection and gridlines above need no download, which is why
> this lab stops there.

### YOUR TURN 4

Map the **decline**, not the depth. For each well in `levels`, compute the change
in depth to water from its first to its last measurement, then map those 80
wells with a **diverging** colormap centred on zero.

A diverging map is right here because zero is meaningful: negative is recovery,
positive is decline, and the reader needs to see which side of zero a point is on.

In [ ]:
change = (
    levels.sort_values("date").groupby("site_no")["depth_to_water_ft"]
    .agg(lambda s: s.iloc[-1] - s.iloc[0])
    .rename("change_ft")
)
mapped = wells.merge(change, left_on="site_no", right_index=True)
print(f"{len(mapped)} wells with a change value")

fig, ax = plt.subplots(figsize=(7, 6.5))

# YOUR TURN
sc = ...

ax.set_aspect(1 / np.cos(np.deg2rad(mapped["latitude"].mean())))
fig.colorbar(sc, ax=ax, label="change in depth to water (ft)")
ax.set_xlabel("longitude")
ax.set_ylabel("latitude")
ax.set_title("Water table decline is not evenly distributed across the basin")
plt.tight_layout()
plt.show()

In [ ]:
# CHECK
assert sc is not None, "assign ax.scatter(...) to sc"
arr = sc.get_array()
assert arr is not None, "colour has to carry change_ft"
assert sc.get_cmap().name in {"RdBu", "RdBu_r", "coolwarm", "bwr", "seismic", "PuOr"}, \
    f"{sc.get_cmap().name!r} is not diverging — zero has to sit in the middle"
lo, hi = sc.get_clim()
assert abs(lo + hi) < 1e-6, (
    f"colour limits {lo:.1f} to {hi:.1f} are not symmetric about zero — "
    "pass vmin=-v, vmax=+v or the neutral colour lands somewhere arbitrary"
)
print(f"{len(arr)} wells, {sc.get_cmap().name}, limits {lo:.0f} to {hi:.0f}. Correct.")

**That symmetry check is the point of the exercise.** A diverging colormap whose
limits are −6 to +111 puts its neutral colour at +52 ft of decline, so wells that
have dropped fifty feet appear white — "unchanged". The colormap has to be
centred on zero or it says the opposite of what you mean.

---

## Part 5 — Saving

The figure on your screen and the figure in the document are not the same object.

In [ ]:
OUT = ROOT / "_run" / "week08_figures"
OUT.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(figsize=(6.5, 4))       # size in INCHES
ax.plot(climatology.index, climatology.values, color="#AB0520", lw=2, marker="o")
ax.set_xlabel("month")
ax.set_ylabel("mean daily discharge (cfs)")
ax.set_title("Sabino Creek monthly climatology, 2005-2024")
ax.grid(alpha=0.3)

fig.savefig(OUT / "climatology.png", dpi=300, bbox_inches="tight")
fig.savefig(OUT / "climatology.pdf", bbox_inches="tight")
plt.show()

for f in sorted(OUT.iterdir()):
    print(f"{f.name:20s} {f.stat().st_size / 1024:8.1f} kB")

The four arguments that matter:

- **`figsize`** is in inches, and it sets how big the text is *relative to* the
  data. Scaling a figure down in Word shrinks the fonts with it; setting
  `figsize` to the final printed width does not.
- **`dpi=300`** for raster output. 72 looks fine on screen and terrible on paper.
- **`bbox_inches="tight"`** crops the whitespace, and stops a long y-label being
  cut off.
- **`.pdf` or `.svg`** for anything going into a paper — vector, so it stays
  sharp at any zoom, and usually smaller than the PNG.

### YOUR TURN 5

Produce one figure you would put in front of an audience: the basin-wide record.

Requirements the check enforces:

- two panels sharing the x axis
- top: median depth to water across all wells, by year
- bottom: how many wells were measured in each year
- both y axes labelled, the top one inverted, a title that states a finding
- saved to `OUT / "basin_summary.png"` at 300 dpi

In [ ]:
annual = levels.assign(year=levels["date"].dt.year)
median_depth = annual.groupby("year")["depth_to_water_ft"].median()
n_wells = annual.groupby("year")["site_no"].nunique()

# YOUR TURN
fig, axes = ...

In [ ]:
# CHECK
saved = OUT / "basin_summary.png"
assert saved.exists(), f"save the figure to {saved}"
assert saved.stat().st_size > 40_000, "that file is too small for 300 dpi — check the dpi"
assert len(axes) == 2, f"expected 2 panels, got {len(axes)}"
assert axes[0].get_shared_x_axes().joined(axes[0], axes[1]), "share the x axis"
assert axes[0].yaxis_inverted(), "invert the top y axis — depth counts downward"
assert axes[0].get_ylabel() and axes[1].get_ylabel(), "label both y axes"
assert len(axes[0].get_title() or fig._suptitle.get_text()) > 20, "give it a real title"
print(f"saved {saved.name}, {saved.stat().st_size / 1024:.0f} kB. Correct.")

**Look at what the bottom panel does to the top one.** The number of wells
measured per year is not decoration — it is the caveat, and here it changes the
reading entirely.

Between 2007 and 2008 the median depth jumps from about 71 ft to about 242 ft.
The aquifer did not drop 170 feet in a year. The well count went from three to
five, and the three happened to be shallow ones. **That entire feature is a
sampling artifact**, and it is invisible without the bottom panel.

The genuine signal is narrower: among the five wells measured consistently since
2011, the median has risen about 32 ft. That is a real recovery in a real set of
wells, and it is a much smaller claim than "the basin recovered".

Putting the sample size under the trend is how you make a figure that is honest
about what it can support. It is also the difference between a figure that
survives review and one that comes back.

---

## Before you leave

1. *Kernel → Restart Kernel and Run All Cells*
2. Fix anything that breaks
3. Save

## What's due

- **HW 6 — Timeseries aggregation**, Wednesday 10/14 at 11:59pm
- **Project 2 analysis, part 1**

## Next week

Statistics and regression: fitting a trend, and saying honestly how much you
believe it. Then water chemistry.

## Stuck?

- A figure that doesn't appear usually means you forgot `plt.show()`, or the
  cell ended with something else.
- Overlapping labels: `plt.tight_layout()`, or `fig.subplots_adjust()` for fine
  control.
- `ax.legend()` showing nothing means no artist has a `label=`.
- Text cut off in a saved file means you left out `bbox_inches="tight"`.
- Office hours: Tuesdays 1:00–2:00pm, Harshbarger 322B.